we're building the agentic loop to understand how the model processes information, uses tools, and arrives at a final response

In [1]:
!pip install -q "smolagents[openai]" requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 9.4 MB/s eta 0:00:00


In [2]:
import os
import requests
from getpass import getpass

from smolagents import CodeAgent, OpenAIModel, FinalAnswerTool, tool

os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter API Key: ")

model = OpenAIModel(
    model_id="openrouter/free",
    api_base="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    max_tokens=1024,
    temperature=0.3,
)

Enter your OpenRouter API Key: ··········


In [3]:
@tool
def get_weather(location: str) -> str:
    """Get the current weather for a location.
    Args:
        location: City or place name, for example 'London', 'Delhi', or 'New York'.
    """
    try:
        url = f"https://wttr.in/{location}?format=j1"
        response = requests.get(url, timeout=20)
        response.raise_for_status()

        data = response.json()
        current = data["current_condition"][0]

        temp_c = current["temp_C"]
        feels_like_c = current["FeelsLikeC"]
        description = current["weatherDesc"][0]["value"]
        humidity = current["humidity"]
        wind_kmph = current["windspeedKmph"]

        return (
            f"Current weather in {location}: {description}. "
            f"Temperature: {temp_c}°C, feels like {feels_like_c}°C. "
            f"Humidity: {humidity}%. Wind speed: {wind_kmph} km/h."
        )

    except Exception as e:
        return f"Could not fetch weather for {location}. Error: {str(e)}"


final_answer = FinalAnswerTool()

agent = CodeAgent(
    model=model,
    tools=[
        final_answer,
        get_weather,
    ],
    max_steps=4,
)

agent.run("What is the current weather in London?")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the current weather in London?                                                                          │
│                                                                                                                 │
╰─ OpenAIModel - openrouter/free ─────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  weather = get_weather(location="London")                                                                         
  print(weather)                                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Current weather in London: Sunny. Temperature: 18°C, feels like 18°C. Humidity: 77%. Wind speed: 5 km/h.

Out: None

[Step 1: Duration 16.87 seconds| Input tokens: 2,033 | Output tokens: 80]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Sunny. Temperature: 18°C, feels like 18°C. Humidity: 77%. Wind speed: 5 km/h.")                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Sunny. Temperature: 18°C, feels like 18°C. Humidity: 77%. Wind speed: 5 km/h.

[Step 2: Duration 2.25 seconds| Input tokens: 4,157 | Output tokens: 243]

'Sunny. Temperature: 18°C, feels like 18°C. Humidity: 77%. Wind speed: 5 km/h.'

In [4]:
agent.run("What is the current weather in Delhi?")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the current weather in Delhi?                                                                           │
│                                                                                                                 │
╰─ OpenAIModel - openrouter/free ─────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  weather = get_weather(location="Delhi")                                                                          
  print(weather)                                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Current weather in Delhi: Haze. Temperature: 40°C, feels like 40°C. Humidity: 20%. Wind speed: 16 km/h.

Out: None

[Step 1: Duration 5.99 seconds| Input tokens: 2,204 | Output tokens: 107]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("The current weather in Delhi is: Haze, Temperature: 40°C (feels like 40°C), Humidity: 20%, Wind    
  speed: 16 km/h.")                                                                                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: The current weather in Delhi is: Haze, Temperature: 40°C (feels like 40°C), Humidity: 20%, Wind 
speed: 16 km/h.

[Step 2: Duration 4.84 seconds| Input tokens: 4,591 | Output tokens: 255]

'The current weather in Delhi is: Haze, Temperature: 40°C (feels like 40°C), Humidity: 20%, Wind speed: 16 km/h.'